# Assignment 2

AME 5052 - Applied Machine Learning Lab

Run the cells from top to bottom. The examples use built-in scikit-learn datasets. Output tables print below each cell and plots display inline.

In [ ]:
# AME 5052 - Applied Machine Learning Lab - Assignment 2
# Run: python Assignment_2.py
# Required packages: numpy, pandas, matplotlib, scikit-learn, scipy

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
# Use the notebook plotting backend
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram

OUT_DIR = Path("assignment2_outputs")
OUT_DIR.mkdir(exist_ok=True)

## Distance/similarity, K-means, hierarchical clustering and dendrograms

In [ ]:
print("\n2(a) Distances, K-means, and hierarchical clustering")
iris = load_iris()
X_scaled = StandardScaler().fit_transform(iris.data)
print("Euclidean distances for the first three rows:")
print(np.round(euclidean_distances(X_scaled[:3]), 3))
print("Cosine similarities for the first three rows:")
print(np.round(cosine_similarity(X_scaled[:3]), 3))

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
print(f"K-means silhouette score: {silhouette_score(X_scaled, kmeans_labels):.3f}")

linkage_scores = []
for method in ["ward", "complete", "average", "single"]:
    model = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    linkage_scores.append({"Linkage": method, "Silhouette": score})
    print(f"{method.title()} linkage silhouette score: {score:.3f}")

# A smaller sample keeps the dendrogram readable.
Z = linkage(X_scaled[:50], method="ward", metric="euclidean")
plt.figure(figsize=(8, 4.5))
dendrogram(Z, truncate_mode="lastp", p=18, leaf_rotation=45, leaf_font_size=8)
plt.title("Hierarchical Clustering Dendrogram (50 Iris Samples)")
plt.xlabel("Merged sample groups")
plt.ylabel("Euclidean distance")
plt.tight_layout()
plt.savefig(OUT_DIR / "hierarchical_dendrogram.png", dpi=150)
plt.show()

plt.figure(figsize=(5.5, 4))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=kmeans_labels, alpha=0.8)
plt.xlabel("Standardized feature 1")
plt.ylabel("Standardized feature 2")
plt.title("K-means Cluster Assignments")
plt.tight_layout()
plt.savefig(OUT_DIR / "kmeans_clusters.png", dpi=150)
plt.show()

## Ensemble learning: Random Forest bagging and Gradient Boosting

In [ ]:
print("\n2(b) Ensemble methods: Random Forest and Gradient Boosting")
cancer = load_breast_cancer()
X = cancer.data
y = (cancer.target == 0).astype(int)  # 1 = malignant
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
ensemble_models = {
    "Random Forest (bagging)": RandomForestClassifier(n_estimators=200, max_features="sqrt", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
}
ensemble_rows = []
for name, model in ensemble_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    ensemble_rows.append({"Model": name, "Accuracy": acc, "Malignant F1": f1})
    print(f"{name}: accuracy={acc:.3f}, malignant-class F1={f1:.3f}")
ensemble_results = pd.DataFrame(ensemble_rows)
plt.figure(figsize=(6, 3.8))
plt.bar(ensemble_results["Model"], ensemble_results["Accuracy"])
plt.ylim(0, 1.05)
plt.ylabel("Test accuracy")
plt.title("Ensemble Model Comparison")
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "ensemble_comparison.png", dpi=150)
plt.show()

## Gaussian Mixture Model (GMM) clustering

In [ ]:
print("\n2(c) Gaussian Mixture Model")
iris = load_iris()
X_scaled = StandardScaler().fit_transform(iris.data)
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=42, n_init=5)
gmm_labels = gmm.fit_predict(X_scaled)
print(f"GMM silhouette score: {silhouette_score(X_scaled, gmm_labels):.3f}")
print(f"GMM AIC: {gmm.aic(X_scaled):.2f}")
print(f"GMM BIC: {gmm.bic(X_scaled):.2f}")
print("Membership probabilities for the first five samples:")
print(np.round(gmm.predict_proba(X_scaled[:5]), 3))
print(f"Adjusted Rand Index vs known Iris classes (evaluation only): {adjusted_rand_score(iris.target, gmm_labels):.3f}")

pca_for_plot = PCA(n_components=2)
X_2d = pca_for_plot.fit_transform(X_scaled)
plt.figure(figsize=(5.5, 4))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=gmm_labels, alpha=0.8)
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("GMM Clusters Shown in 2D PCA Space")
plt.tight_layout()
plt.savefig(OUT_DIR / "gmm_clusters_pca_view.png", dpi=150)
plt.show()

## Principal Component Analysis (PCA) and visualization

In [ ]:
print("\n2(d) Principal Component Analysis")
wine = load_wine()
X_scaled = StandardScaler().fit_transform(wine.data)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_
print(f"Variance explained by PC1: {explained[0]:.3f}")
print(f"Variance explained by PC2: {explained[1]:.3f}")
print(f"Total variance retained by two components: {explained.sum():.3f}")
plt.figure(figsize=(6, 4.5))
for class_id, class_name in enumerate(wine.target_names):
    mask = wine.target == class_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=class_name, alpha=0.8)
plt.xlabel(f"PC1 ({explained[0]:.1%} variance)")
plt.ylabel(f"PC2 ({explained[1]:.1%} variance)")
plt.title("Wine Dataset Visualized with PCA")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "pca_wine_visualization.png", dpi=150)
plt.show()
print(f"Plots saved in: {OUT_DIR.resolve()}")